In [1]:
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from sklearn.model_selection import train_test_split

import random
import os
from PIL import Image

# Check if GPU is available and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


Transform the images into 256x256 pixels, for a consistent input size for the model.

In [2]:
# Define the input and output directories
input_dir = "data"
output_dir = "data_resized"
model_dir = "models"

if not os.path.exists(model_dir):
    os.makedirs(model_dir, exist_ok=True)

if os.path.exists(output_dir):
    print(f"Output directory '{output_dir}' already exists. Please remove it or choose a different name.")
else:
    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Loop through each folder and image in the input directory
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith((".png", ".jpg", ".jpeg")):  # Check for image files
                input_path = os.path.join(root, file)
                
                # Open the image and resize it
                with Image.open(input_path) as img:
                    img_resized = img.resize((256, 256))
                    
                    # Create the corresponding output path
                    relative_path = os.path.relpath(root, input_dir)
                    output_folder = os.path.join(output_dir, relative_path)
                    os.makedirs(output_folder, exist_ok=True)
                    output_path = os.path.join(output_folder, file)
                    
                    # Save the resized image
                    img_resized.save(output_path)

    print("Images have been resized and saved to 'data_resized'.")

Output directory 'data_resized' already exists. Please remove it or choose a different name.


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiTaskCNN(nn.Module):
    def __init__(self):
        super(MultiTaskCNN, self).__init__()

        # Improved Feature Extractor
        self.features = nn.Sequential(
            # Block 1: 256x256 -> 128x128
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 2: 128x128 -> 64x64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 3: 64x64 -> 32x32
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 4: 32x32 -> 16x16
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )

        # Global Average Pooling to reduce parameter count
        self.gap = nn.AdaptiveAvgPool2d(1)

        # Classifier with Dropout
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, 5)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# Re-instantiate the model
if 'device' not in locals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiTaskCNN().to(device)
print(model)

MultiTaskCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1,

We now load the resized images and their corresponding labels into memory. 
The label is the name of the folder that contains the image, which represents the class of the image.

In [4]:
from typing import Optional

class ResizedImageDataset(Dataset):
    def __init__(self, root_dir: str, transform: Optional[transforms.Compose] = None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths: list[str] = []
        self.labels: list[str] = []

        # Loop through the directory to collect image paths and labels
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(root, file))
                    self.labels.append(os.path.basename(root))  # Folder name as label

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        img_path = self.image_paths[idx]
        label: str = self.labels[idx]
        
        # Open the image
        with Image.open(img_path) as img:
            if self.transform:
                img = self.transform(img)
        
        if not isinstance(img, torch.Tensor):
            img = transforms.ToTensor()(img)  # Ensure the image is a tensor

        return img, label

# Define transformations for the dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Normalize to [-1, 1]
])

# Create the dataset
dataset = ResizedImageDataset(output_dir, transform=transform)

# Example: Load the first image and label
img, label = dataset[random.randint(0, len(dataset)-1)]
print(f"Image shape: {img.shape}, Label: {label}")

Image shape: torch.Size([3, 256, 256]), Label: 0-noDR


In [5]:
def divide_train_val(dataset: ResizedImageDataset, val_size: float) -> tuple[ResizedImageDataset, ResizedImageDataset]:
    # Group images and labels by class
    class_to_images = {}
    for img_path, label in zip(dataset.image_paths, dataset.labels):
        if label not in class_to_images:
            class_to_images[label] = []
        class_to_images[label].append(img_path)

    train_image_paths = []
    train_labels = []
    val_image_paths = []
    val_labels = []

    # Split each class into training and validation sets
    for label, image_paths in class_to_images.items():
        train_paths, val_paths = train_test_split(image_paths, test_size=val_size / len(dataset), shuffle=False)
        train_image_paths.extend(train_paths)
        train_labels.extend([label] * len(train_paths))
        val_image_paths.extend(val_paths)
        val_labels.extend([label] * len(val_paths))

    # Create training and validation datasets
    train_dataset = ResizedImageDataset(root_dir='.', transform=transform)
    train_dataset.image_paths = train_image_paths
    train_dataset.labels = train_labels

    val_dataset = ResizedImageDataset(root_dir='.', transform=transform)
    val_dataset.image_paths = val_image_paths
    val_dataset.labels = val_labels

    return train_dataset, val_dataset

In [ ]:

# ==================== Configuration ====================
num_epochs = 50
batch_size = 32
learning_rate = 0.001
weight_decay = 1e-5
model_save_interval = 20  # Save model every 20 epochs
model_save_path = os.path.join(model_dir, "multi_task_cnn.pth")

# ==================== Data Loading ====================
# Create DataLoader for training and validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
# Split the dataset into training and validation sets, ensuring each class is represented proportionally

train_dataset, val_dataset = divide_train_val(dataset, val_size=val_size)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# ==================== Training Setup ====================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)  # Reduce LR by 50% every 10 epochs

# ==================== Training Loop ====================
model.train()
model.to(device) # just in case

for epoch in range(num_epochs):
    total_loss = 0.0
    
    for images, labels in train_loader:
        # Move data to device
        images = images.to(device)
        index_labels = torch.tensor([int(label.split('-')[0]) for label in labels], dtype=torch.long).to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, index_labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    # Update learning rate
    scheduler.step()
    
    # Print progress
    avg_loss = total_loss / len(train_loader)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, LR: {current_lr:.6f}")
    
    # Save model every model_save_interval epochs
    if (epoch + 1) % model_save_interval == 0:
        torch.save(model.state_dict(), model_save_path)
        print(f"Model saved to {model_save_path}")

print("Training completed!")

Epoch [1/50], Loss: 1.4543, LR: 0.001000
Epoch [2/50], Loss: 1.4233, LR: 0.001000
Epoch [3/50], Loss: 1.4182, LR: 0.001000
Epoch [4/50], Loss: 1.4052, LR: 0.001000
Epoch [5/50], Loss: 1.4058, LR: 0.001000
Epoch [6/50], Loss: 1.3872, LR: 0.001000
Epoch [7/50], Loss: 1.3966, LR: 0.001000
Epoch [8/50], Loss: 1.3869, LR: 0.001000
Epoch [9/50], Loss: 1.4010, LR: 0.001000
Epoch [10/50], Loss: 1.3753, LR: 0.000500


In [ ]:
# Save the model's state dictionary
model_save_path = os.path.join(model_dir, f"multi_task_cnn_final_{epoch}_{loss:.4f}_.pth")
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

Model saved to models/multi_task_cnn_final_49_1.5464_.pth


In [ ]:
from sklearn.metrics import accuracy_score

if model_save_path not in locals():
    model_save_path = os.path.join(model_dir, "multi_task_cnn_final_49_1.5464_.pth")  # Placeholder for accuracy
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# Load the model's state dictionary
model.load_state_dict(torch.load(model_save_path, map_location=device))
model.to(device)  # Move model to the device

# Initialize lists to store predictions and true labels
preds = []
true_labels = []

# Set the model to evaluation mode
model.eval()

# Perform validation
with torch.no_grad():
    for images, labels in val_loader:
        # Move data to the device
        images = images.to(device)
        index_labels = torch.tensor([int(label.split('-')[0]) for label in labels], dtype=torch.long).to(device)

        # Forward pass
        out = model(images)

        # Collect predictions
        preds.extend(torch.argmax(out, dim=1).cpu().numpy())
        true_labels.extend(index_labels.cpu().numpy())

# Calculate accuracy for multi-class classification
accuracy = accuracy_score(true_labels, preds)

print(f"Validation Accuracy: {accuracy:.4f}")

Validation Accuracy: 0.3730


In [ ]:
import os

# Get a random image and its label from the dataset
random_idx = random.randint(0, len(dataset) - 1)
img, label = dataset[random_idx]

# Move the image to the device and add a batch dimension
img = img.unsqueeze(0).to(device)

# Set the model to evaluation mode
model.eval()

# Perform inference
with torch.no_grad():
    output = model(img)
    predicted_class = torch.argmax(output, dim=1).item()

# Map the predicted class index to the folder name
class_mapping = {0: "0-noDR", 1: "1-mild", 2: "2-moderate", 3: "3-severe", 4: "4-proliferativeDR"}
predicted_label = class_mapping[predicted_class]

# Print the results
print(f"True Label: {label}, Predicted Label: {predicted_label}")

# Verify if the image is in the correct folder
image_path = dataset.image_paths[random_idx]
print(f"Image Path: {image_path}")
print(f"Is the image in the correct folder? {'Yes' if predicted_label in image_path else 'No'}")

True Label: 0-noDR, Predicted Label: 0-noDR
Image Path: data_resized/0-noDR/2337_right.jpeg
Is the image in the correct folder? Yes
